# Clasificador de piezas — tanda 2D y 3D (cámara principal desde arriba + secundaria)

Entrena **dos modelos seguidos** que dicen **qué pieza** hay en cada casilla (13 clases: 6 piezas × 2 colores + vacía) y los compara:

- **2D** (`ampliado`): la casilla vista desde arriba con un margen alrededor.
- **3D** (`volumen`): la caja que ocupa la pieza vista con el lente de la cámara.

**Las fotos** salen de `scripts/exportar_zip_colab.py`: la **webcam principal** (ahora puesta desde arriba) tomadas desde su última calibración, tus
casillas corregidas en el probador y las de la **cámara secundaria** que antes estaba arriba (más nítida y con más fotos).

**Cómo se usan:**

- **Se valida y se prueba solo con la principal** (la webcam, que es la que se va a usar): así la nota es la de verdad.
- Las fotos de la secundaria **solo entrenan** y se **degradan** (se achican y se desenfocan un poco) para que se parezcan a las de la webcam.
- Red **preentrenada en ImageNet** (EfficientNet-B0) con aumentaciones de perspectiva, luz y desenfoque.

Al final guarda `clasificador_piezas_2d.pt`, `clasificador_piezas_3d.pt` y `clasificador_piezas.pt` (el que sacó mejor F1 en la prueba).

## 1 · Configuración

In [ ]:
RECORTES = ["ampliado", "volumen"]  # la tanda: 2D y 3D (se entrenan uno tras otro). Para uno solo: ["ampliado"] o ["volumen"]
CAMARA_OBJETIVO = "principal"       # la cámara con la que se va a usar el modelo (la webcam desde arriba); con ella se valida y se mide
USAR_OTRAS_CAMARAS = True           # sumar al entrenamiento las fotos de la otra cámara (la secundaria, más nítida)
PESO_OTRAS_CAMARAS = 0.5            # cuánto pesan esas fotos frente a las de la cámara objetivo
DEGRADAR_OTRAS_CAMARAS = True       # que las fotos de la otra cámara se vean tan poco nítidas como las de la webcam
ARQUITECTURA = "efficientnet_b0"    # "efficientnet_b0", "resnet18" o "convnext_tiny" (más lenta, a veces mejor)
PREENTRENADO = True                 # partir de los pesos de ImageNet (False solo para probar el cuaderno sin internet)
TAMANO = 128                        # lado de la imagen que entra a la red (el backend redimensiona igual)
EPOCAS = 60
PACIENCIA = 15                      # corta si la validación no mejora en tantas épocas
LOTE = 64
LR = 3e-4
SEMILLA = 7
FRACCION_VALIDACION = 0.15
FRACCION_PRUEBA = 0.15
RUTA_ZIP_EN_DRIVE = None  # None = usa el ZIP dataset_piezas*.zip MÁS NUEVO de tu Drive. Para forzar uno: "/content/drive/MyDrive/dataset_piezas_20261023_1950.zip"

CLASES = sorted([f"{c}-{p}" for c in ("white", "black") for p in ("pawn", "knight", "bishop", "rook", "queen", "king")]) + ["vacia"]
# mismo orden que el backend (modelo_piezas.CLASES)

## 2 · Entorno

In [ ]:
import os, io, json, random, zipfile, time, math
from collections import Counter, defaultdict
import numpy as np, pandas as pd, cv2, torch, torchvision
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import v2 as T
import matplotlib.pyplot as plt

EN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # mps: GPU de una Mac
print(f"torch {torch.__version__} · torchvision {torchvision.__version__} · dispositivo: {DISPOSITIVO}")
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU: va a ser MUY lento (horas). En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU; o corrélo en una Mac con chip Apple.")
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

## 3 · Datos: el ZIP exportado desde la app

In [ ]:
CARPETA = "/content/dataset" if EN_COLAB else os.environ.get("CARPETA_DATASET", "dataset")
ruta_zip = os.environ.get("RUTA_ZIP")
if EN_COLAB and not ruta_zip:
    import glob
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)  # remontar: un archivo recién subido a Drive a veces no aparece hasta hacerlo
    mi_unidad = "/content/drive/MyDrive"
    candidatos = sorted(glob.glob(f"{mi_unidad}/dataset_piezas*.zip") + glob.glob(f"{mi_unidad}/*/dataset_piezas*.zip"), key=os.path.getmtime)
    for z in candidatos:
        print(f"  ZIP en tu Drive: {z.replace(mi_unidad + '/', '')} · {os.path.getsize(z) / 1e6:.0f} MB · {time.strftime('%d/%m %H:%M', time.localtime(os.path.getmtime(z)))}")
    if RUTA_ZIP_EN_DRIVE and os.path.exists(RUTA_ZIP_EN_DRIVE):
        ruta_zip = RUTA_ZIP_EN_DRIVE
    elif candidatos:
        ruta_zip = candidatos[-1]  # el más nuevo, aunque se llame «dataset_piezas (1).zip» o esté en una carpeta
    else:
        zips = sorted(os.path.basename(z) for z in glob.glob(f"{mi_unidad}/*.zip"))
        raise FileNotFoundError(
            "No encontré dataset_piezas*.zip en tu Drive (en «Mi unidad» o en una carpeta de primer nivel). Subilo a drive.google.com, "
            "esperá a que termine de subir y volvé a ejecutar esta celda. ZIP que sí hay en Mi unidad: " + (", ".join(zips) or "ninguno")
        )
    print("Usando", ruta_zip, f"({os.path.getsize(ruta_zip) / 1e6:.0f} MB)")
if not os.path.isdir(os.path.join(CARPETA, "ampliado")):
    with zipfile.ZipFile(ruta_zip) as z:
        z.extractall(CARPETA)
datos_todos = pd.read_csv(os.path.join(CARPETA, "metadatos.csv"), dtype=str).fillna("")
datos_todos = datos_todos[datos_todos.clase.isin(CLASES)].copy()
datos_todos.loc[datos_todos.grupo == "", "grupo"] = datos_todos.loc[datos_todos.grupo == "", "captura"]
datos_todos["camara"] = datos_todos.camara.replace("", "desconocida")
print(f"{len(datos_todos)} recortes en total")
display(pd.crosstab([datos_todos.recorte, datos_todos.camara], datos_todos.clase)) if "display" in globals() else print(pd.crosstab([datos_todos.recorte, datos_todos.camara], datos_todos.clase))
if not (datos_todos.camara == CAMARA_OBJETIVO).any():
    raise ValueError(f"El ZIP no tiene fotos de la cámara «{CAMARA_OBJETIVO}»: no hay con qué validar. Exportalo de nuevo con scripts/exportar_zip_colab.py")
n_objetivo = int(((datos_todos.camara == CAMARA_OBJETIVO) & (datos_todos.recorte == RECORTES[0])).sum())
print(f"\n>>> {n_objetivo} recortes de «{CAMARA_OBJETIVO}» y {int(((datos_todos.camara != CAMARA_OBJETIVO) & (datos_todos.recorte == RECORTES[0])).sum())} de otras cámaras en este ZIP ({ruta_zip}).")
if n_objetivo < 500:
    print("⚠ PARECE EL ZIP VIEJO: el nuevo trae unos 800 recortes de la principal y unos 1350 de la secundaria. Subí el ZIP nuevo a Drive y poné su ruta en RUTA_ZIP_EN_DRIVE.")


## 4 · División por captura

Entrenamiento, validación y prueba se separan por **grupo** (el instante de la foto), así la nota es honesta. **Solo los grupos de la cámara objetivo**
se reparten entre las tres partes; **todas** las fotos de las otras cámaras van a entrenamiento (si no, se perdería parte de ellas sin motivo).

In [ ]:
def dividir(datos):
    """`(entreno, valid, prueba)`: validación y prueba salen solo de la cámara objetivo (estratificado por clase y por grupo)."""
    azar = random.Random(SEMILLA)
    objetivo = datos[datos.camara == CAMARA_OBJETIVO]
    clase_del_grupo = objetivo.groupby("grupo").clase.agg(lambda c: Counter(c).most_common(1)[0][0])
    destino = {}
    for clase, grupos in clase_del_grupo.groupby(clase_del_grupo):  # cada clase reparte sus grupos
        lista = sorted(grupos.index); azar.shuffle(lista)
        n_prueba = max(1, round(len(lista) * FRACCION_PRUEBA)) if len(lista) >= 3 else 0
        n_valid = max(1, round(len(lista) * FRACCION_VALIDACION)) if len(lista) >= 3 else 0
        for i, g in enumerate(lista):
            destino[g] = "prueba" if i < n_prueba else "valid" if i < n_prueba + n_valid else "entreno"
    parte = objetivo.grupo.map(destino)
    otras = datos[(datos.camara != CAMARA_OBJETIVO)] if USAR_OTRAS_CAMARAS else datos.iloc[0:0]
    return pd.concat([objetivo[parte == "entreno"], otras]), objetivo[parte == "valid"], objetivo[parte == "prueba"]

## 5 · Imágenes, aumentaciones y modelo

Se lee igual que en el backend (`cv2`, BGR → RGB, redimensionado lineal, de 0 a 1). Las aumentaciones simulan otra posición de la cámara (perspectiva,
rotación, inclinación, escala, encuadre), otra luz y desenfoque. **No** se cambia el tono ni se pasa a gris: el color de la pieza y la marca rojiza de
los alfiles tienen que llegar intactos. Las fotos de la otra cámara (más nítidas) pasan antes por `degradar_webcam`.

In [ ]:
def leer(ruta):
    imagen = cv2.imread(os.path.join(CARPETA, ruta))
    imagen = cv2.resize(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB), (TAMANO, TAMANO), interpolation=cv2.INTER_LINEAR)
    return torch.from_numpy(imagen).permute(2, 0, 1).float() / 255.0

AUMENTAR = T.Compose([
    T.RandomResizedCrop(TAMANO, scale=(0.72, 1.0), ratio=(0.85, 1.18), antialias=True),
    T.RandomPerspective(distortion_scale=0.35, p=0.6),
    T.RandomAffine(degrees=12, translate=(0.06, 0.06), shear=(-8, 8, -4, 4)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.2, hue=0.0),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.6))], p=0.3),
    T.RandomErasing(p=0.2, scale=(0.02, 0.08)),
])

def degradar_webcam(x):
    """Una foto nítida (la cámara de arriba de antes) se achica y se vuelve a agrandar y se desenfoca un poco: queda como las de la webcam."""
    lado = max(16, int(TAMANO * random.uniform(0.62, 0.9)))
    x = F.interpolate(x[None], size=(lado, lado), mode="area")
    x = F.interpolate(x, size=(TAMANO, TAMANO), mode="bicubic", align_corners=False)[0].clamp(0, 1)
    return T.functional.gaussian_blur(x, kernel_size=5, sigma=random.uniform(0.3, 0.9))

class Recortes(Dataset):
    def __init__(self, filas, aumentar):
        self.rutas, self.y, self.aumentar = list(filas.archivo), [CLASES.index(c) for c in filas.clase], aumentar
        self.degradar = [aumentar and DEGRADAR_OTRAS_CAMARAS and cam != CAMARA_OBJETIVO for cam in filas.camara]
        self.cache = {}
    def __len__(self):
        return len(self.rutas)
    def __getitem__(self, i):
        if i not in self.cache:
            self.cache[i] = leer(self.rutas[i])
        x = self.cache[i]
        if self.degradar[i]:
            x = degradar_webcam(x)
        return (AUMENTAR(x).clamp(0, 1) if self.aumentar else x), self.y[i]

def muestreador(filas):
    """Cada clase sale con la misma frecuencia; las fotos de la otra cámara pesan PESO_OTRAS_CAMARAS."""
    por_clase = filas.clase.value_counts()
    pesos = [(1.0 / por_clase[c]) * (1.0 if cam == CAMARA_OBJETIVO else PESO_OTRAS_CAMARAS) for c, cam in zip(filas.clase, filas.camara)]
    return WeightedRandomSampler(pesos, num_samples=len(filas), replacement=True)

class Clasificador(nn.Module):
    """Red preentrenada con la normalización de ImageNet ADENTRO: recibe RGB de 0 a 1, como lo prepara el backend."""
    def __init__(self, red):
        super().__init__()
        self.red = red
        self.register_buffer("media", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("desvio", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.red((x - self.media) / self.desvio)

def crear_red(nombre, clases):
    pesos = "DEFAULT" if PREENTRENADO else None
    if nombre == "resnet18":
        red = torchvision.models.resnet18(weights=pesos); red.fc = nn.Linear(red.fc.in_features, clases); cabeza = red.fc
    elif nombre == "efficientnet_b0":
        red = torchvision.models.efficientnet_b0(weights=pesos); red.classifier[1] = nn.Linear(red.classifier[1].in_features, clases); cabeza = red.classifier
    elif nombre == "convnext_tiny":
        red = torchvision.models.convnext_tiny(weights=pesos); red.classifier[2] = nn.Linear(red.classifier[2].in_features, clases); cabeza = red.classifier
    else:
        raise ValueError(nombre)
    return Clasificador(red), cabeza

def evaluar(red, cargador_eval, espejo=True):
    red.eval(); reales, probs = [], []
    with torch.no_grad(), torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == "cuda"):
        for x, y in cargador_eval:
            x = x.to(DISPOSITIVO)
            p = torch.softmax(red(x).float(), 1)
            if espejo:  # promedio con la imagen espejada (TTA)
                p = (p + torch.softmax(red(torch.flip(x, dims=[3])).float(), 1)) / 2
            probs.append(p.cpu()); reales += list(y)
    probs = torch.cat(probs).numpy() if probs else np.zeros((0, len(CLASES)))
    return np.array(reales), probs

def f1_macro(reales, predichas):
    f1s = []
    for k in range(len(CLASES)):
        vp = np.sum((predichas == k) & (reales == k)); fp = np.sum((predichas == k) & (reales != k)); fn = np.sum((predichas != k) & (reales == k))
        if vp + fn:
            f1s.append(2 * vp / max(1, 2 * vp + fp + fn))
    return float(np.mean(f1s)) if f1s else 0.0

# vista previa: cómo quedan las fotos de la otra cámara tras degradarlas (arriba, tal cual; abajo, degradadas)
otras = datos_todos[(datos_todos.recorte == RECORTES[0]) & (datos_todos.camara != CAMARA_OBJETIVO)]
if len(otras) >= 8:
    muestra = [leer(r) for r in otras.sample(8, random_state=SEMILLA).archivo]
    plt.figure(figsize=(12, 3.4)); plt.imshow(torchvision.utils.make_grid(muestra + [degradar_webcam(x) for x in muestra], nrow=8).permute(1, 2, 0)); plt.axis("off")
    plt.title("fotos de la otra cámara: originales (arriba) y degradadas a la nitidez de la webcam (abajo)"); plt.show()

## 6 · Entrenar y probar cada variante

AdamW con calentamiento y coseno, *label smoothing* y precisión mixta. Las 2 primeras épocas solo se entrena la cabeza (la parte nueva). Se guarda la época
con mejor **F1 macro** en validación (pesa igual cada pieza, aunque haya muchas más casillas vacías). Luego se mide en la **prueba**: fotos de la webcam
que el modelo nunca vio.

In [ ]:
def entrenar_variante(recorte):
    """Entrena con el recorte `recorte` (2D `ampliado` o 3D `volumen`) y devuelve el modelo con sus notas de prueba."""
    print(f"\n{'=' * 70}\n  {'2D (ampliado)' if recorte == 'ampliado' else '3D (volumen)'}\n{'=' * 70}")
    random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
    datos = datos_todos[datos_todos.recorte == recorte]
    entreno, valid, prueba = dividir(datos)
    print(f"entrenamiento {len(entreno)} ({int((entreno.camara != CAMARA_OBJETIVO).sum())} de otras cámaras) · validación {len(valid)} · prueba {len(prueba)} (las dos últimas solo «{CAMARA_OBJETIVO}»)")
    print(pd.DataFrame({"entreno": entreno.clase.value_counts(), "valid": valid.clase.value_counts(), "prueba": prueba.clase.value_counts()}).fillna(0).astype(int))
    trabajadores = 2 if EN_COLAB else 0
    cargador = DataLoader(Recortes(entreno, True), batch_size=LOTE, sampler=muestreador(entreno), num_workers=trabajadores)
    cargador_valid = DataLoader(Recortes(valid, False), batch_size=LOTE, num_workers=trabajadores)
    cargador_prueba = DataLoader(Recortes(prueba, False), batch_size=LOTE, num_workers=trabajadores)

    modelo, cabeza = crear_red(ARQUITECTURA, len(CLASES))
    modelo = modelo.to(DISPOSITIVO)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=0.05 if ARQUITECTURA == "convnext_tiny" else 1e-4)
    pasos = EPOCAS * max(1, len(cargador)); calentamiento = 2 * max(1, len(cargador))
    planificador = torch.optim.lr_scheduler.LambdaLR(optimizador, lambda s: min(1, (s + 1) / calentamiento) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / pasos))))
    perdida_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    escalador = torch.amp.GradScaler(enabled=DISPOSITIVO == "cuda")
    historial, mejor, mejor_estado, sin_mejora = [], -1.0, None, 0
    for epoca in range(1, EPOCAS + 1):
        solo_cabeza = epoca <= 2
        for nombre, p in modelo.red.named_parameters():
            p.requires_grad = (not solo_cabeza) or any(p is q for q in cabeza.parameters())
        modelo.train(); inicio = time.time(); suma, n = 0.0, 0
        for x, y in cargador:
            x, y = x.to(DISPOSITIVO), y.to(DISPOSITIVO)
            optimizador.zero_grad(set_to_none=True)
            with torch.autocast(DISPOSITIVO, enabled=DISPOSITIVO == "cuda"):
                perdida = perdida_fn(modelo(x), y)
            escalador.scale(perdida).backward(); escalador.step(optimizador); escalador.update(); planificador.step()
            suma += perdida.item() * len(y); n += len(y)
        reales, probs = evaluar(modelo, cargador_valid, espejo=False)
        exactitud = float((probs.argmax(1) == reales).mean()) if len(reales) else 0.0
        f1 = f1_macro(reales, probs.argmax(1))
        historial.append({"epoca": epoca, "perdida": suma / max(1, n), "exactitud": exactitud, "f1": f1})
        print(f"época {epoca:3d} · pérdida {suma / max(1, n):.3f} · validación: exactitud {exactitud:.3f}, F1 macro {f1:.3f} · {time.time() - inicio:.0f} s")
        if f1 > mejor:
            mejor, sin_mejora = f1, 0
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora >= PACIENCIA:
                print(f"Sin mejora en {PACIENCIA} épocas: corto."); break
    modelo.load_state_dict(mejor_estado)
    h = pd.DataFrame(historial)
    plt.figure(figsize=(10, 3)); plt.subplot(1, 2, 1); plt.plot(h.epoca, h.perdida); plt.title(f"pérdida ({recorte})")
    plt.subplot(1, 2, 2); plt.plot(h.epoca, h.exactitud, label="exactitud"); plt.plot(h.epoca, h.f1, label="F1 macro"); plt.legend(); plt.title("validación"); plt.show()

    reales, probs = evaluar(modelo, cargador_prueba, espejo=True)
    predichas = probs.argmax(1)
    exactitud_prueba = float((predichas == reales).mean()) if len(reales) else float("nan")
    f1_prueba = f1_macro(reales, predichas)
    con_pieza = reales != CLASES.index("vacia")
    exactitud_piezas = float((predichas[con_pieza] == reales[con_pieza]).mean()) if con_pieza.any() else float("nan")
    color = lambda k: CLASES[k].split("-")[0]
    color_mal = int(sum(1 for r, p in zip(reales, predichas) if CLASES[r] != "vacia" and CLASES[p] != "vacia" and color(r) != color(p)))
    print(f"PRUEBA {recorte} ({len(reales)} recortes de «{CAMARA_OBJETIVO}»): exactitud {exactitud_prueba:.3f} · solo piezas {exactitud_piezas:.3f} · F1 macro {f1_prueba:.3f} · color equivocado: {color_mal}")
    tabla = pd.DataFrame([{"clase": c, "fotos": int(np.sum(reales == k)), "aciertos": int(np.sum((reales == k) & (predichas == k)))} for k, c in enumerate(CLASES)])
    tabla["precisión"] = (tabla.aciertos / tabla.fotos.clip(lower=1)).round(3)
    print(tabla.to_string(index=False))
    matriz = np.zeros((len(CLASES), len(CLASES)), int)
    for r, p in zip(reales, predichas):
        matriz[r, p] += 1
    plt.figure(figsize=(8, 7)); plt.imshow(matriz, cmap="Blues"); plt.xticks(range(len(CLASES)), CLASES, rotation=90); plt.yticks(range(len(CLASES)), CLASES)
    for i in range(len(CLASES)):
        for j in range(len(CLASES)):
            if matriz[i, j]:
                plt.text(j, i, matriz[i, j], ha="center", va="center", color="white" if matriz[i, j] > matriz.max() / 2 else "black", fontsize=8)
    plt.xlabel("predicha"); plt.ylabel("real"); plt.title(f"matriz de confusión · prueba · {recorte}"); plt.tight_layout(); plt.show()
    return {"recorte": recorte, "modelo": modelo, "exactitud": exactitud_prueba, "exactitud_piezas": exactitud_piezas, "f1": f1_prueba,
            "f1_validacion": mejor, "fotos_prueba": int(len(reales)), "color_mal": color_mal, "prueba": prueba}

resultados = [entrenar_variante(r) for r in RECORTES]

## 7 · Comparar y exportar para el backend

Se guarda cada modelo como **TorchScript** dentro del mismo formato de checkpoint que lee el backend (`backend/servicios/vision/piezas.py`): `clases`,
`tamano`, `recorte` y la red completa (con la normalización adentro). Antes de guardar se comprueba que la red exportada da lo mismo que la original.
`clasificador_piezas.pt` es una copia del que sacó **mejor F1 en la prueba**.

In [ ]:
NOMBRES = {"ampliado": "2d", "volumen": "3d"}
resumen = pd.DataFrame([{"recorte": NOMBRES[r["recorte"]].upper(), "exactitud": round(r["exactitud"], 3), "solo piezas": round(r["exactitud_piezas"], 3),
                         "F1 macro": round(r["f1"], 3), "color mal": r["color_mal"], "fotos prueba": r["fotos_prueba"]} for r in resultados])
print(resumen.to_string(index=False))

def exportar(r):
    modelo_cpu = r["modelo"].float().cpu().eval()
    ejemplo = torch.rand(8, 3, TAMANO, TAMANO)
    # Se comprueba con FOTOS REALES de la prueba (el ruido al azar es un caso raro donde la red duda y cualquier diferencia mínima se amplifica).
    # Se prueba primero la red optimizada (`freeze` funde capas y es más rápida); si no da lo mismo que la original, se usa la sin optimizar, que es idéntica.
    reales_x = torch.stack([Recortes(r["prueba"], False)[i][0] for i in range(min(48, len(r["prueba"])))])
    with torch.no_grad():
        trazada = torch.jit.trace(modelo_cpu, ejemplo[:2])
        p_original = torch.softmax(modelo_cpu(reales_x), 1)
        exportada = None
        for variante, red in (("optimizada", lambda: torch.jit.freeze(trazada)), ("sin optimizar", lambda: trazada)):
            candidata = red()
            p_exportada = torch.softmax(candidata(reales_x), 1)
            diferencia = (p_exportada - p_original).abs().max().item()
            igual_clase = bool((p_exportada.argmax(1) == p_original.argmax(1)).all())
            print(f"  comprobación de la exportación ({variante}): {len(reales_x)} fotos reales · probabilidades distintas hasta {diferencia:.4f} · misma clase en todas: {igual_clase}")
            if diferencia < 2e-2 and igual_clase:
                exportada = candidata
                break
    assert exportada is not None, f"La red exportada no coincide con la original (última diferencia: {diferencia}, misma clase: {igual_clase})"
    buffer = io.BytesIO(); torch.jit.save(exportada, buffer)
    checkpoint = {
        "formato": "torchscript", "torchscript": buffer.getvalue(), "clases": CLASES, "tamano": TAMANO, "recorte": r["recorte"],
        "arquitectura": ARQUITECTURA, "camara_objetivo": CAMARA_OBJETIVO, "torch": torch.__version__, "con_otras_camaras": USAR_OTRAS_CAMARAS,
        "metricas": {"exactitud_prueba": r["exactitud"], "exactitud_solo_piezas": r["exactitud_piezas"], "f1_macro_prueba": r["f1"],
                     "f1_macro_validacion": r["f1_validacion"], "fotos_prueba": r["fotos_prueba"]},
        "fecha": time.strftime("%Y-%m-%d %H:%M"),
    }
    return checkpoint, len(buffer.getvalue())

archivos = []
for r in resultados:
    checkpoint, tamano_bytes = exportar(r)
    nombre = f"clasificador_piezas_{NOMBRES[r['recorte']]}.pt"
    torch.save(checkpoint, nombre); archivos.append(nombre)
    json.dump({k: v for k, v in checkpoint.items() if k != "torchscript"}, open(nombre.replace(".pt", ".json"), "w"), indent=2, ensure_ascii=False)
    r["prueba"].to_csv(f"prueba_{NOMBRES[r['recorte']]}.csv", index=False)  # qué fotos fueron de prueba: para comparar con otro modelo sobre las mismas
    print(f"Guardado {nombre} ({tamano_bytes / 1e6:.1f} MB) · prueba: exactitud {r['exactitud']:.3f}, F1 {r['f1']:.3f}")
mejor_r = max(resultados, key=lambda r: r["f1"])
import shutil
shutil.copy(f"clasificador_piezas_{NOMBRES[mejor_r['recorte']]}.pt", "clasificador_piezas.pt")
shutil.copy(f"clasificador_piezas_{NOMBRES[mejor_r['recorte']]}.json", "clasificador_piezas.json")
archivos.append("clasificador_piezas.pt")
print(f"\nclasificador_piezas.pt = el {NOMBRES[mejor_r['recorte']].upper()} (mejor F1 en la prueba: {mejor_r['f1']:.3f})")
if EN_COLAB:
    from google.colab import files
    if os.path.isdir("/content/drive/MyDrive"):
        for a in archivos + ["clasificador_piezas.json"]:
            shutil.copy(a, f"/content/drive/MyDrive/{a}")
        print("Copias en Drive: MyDrive/" + ", ".join(archivos))
    for a in archivos:
        try:
            files.download(a)
        except Exception as error:  # desde VS Code la descarga directa no funciona: quedan las copias en Drive
            print("No se pudo descargar directo", a, "(", error, "): bajalo de Drive")

## 8 · Instalarlo en el backend

1. Guardá una copia del modelo actual: `training/checkpoints/clasificador_piezas.pt` → por ejemplo `clasificador_piezas.anterior.pt`.
2. Copiá el que quieras usar a `training/checkpoints/clasificador_piezas.pt`: `clasificador_piezas.pt` (el mejor) o `clasificador_piezas_2d.pt` / `clasificador_piezas_3d.pt`.
3. Reiniciá el backend. En el *Probador* tiene que aparecer «piezas (recorte ampliado)» (2D) o «(recorte volumen)» (3D). Las vistas «Tablero plano» y «Cajas 3D»
   del probador siguen al modelo: el 3D usa la caja de la pieza con el lente de la cámara.

**3D:** conviene solo si el lente de la cámara principal está bien calibrado (la pantalla «Lente y altura»); si no, la caja queda corrida y el 2D rinde mejor.

El backend usa `torch 2.14`: un modelo exportado con una versión de torch igual o más vieja se carga sin problema (la versión queda anotada en el checkpoint, campo `torch`).